# [5.3] Mamba from Scratch - Exercises

Implement the selective-scan core first. Recurrent, associative, chunked, and cached inference paths should agree numerically.

In [ ]:
import sys
from pathlib import Path

import torch as t

chapter = "chapter5_modern_architectures"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_mamba_from_scratch.tests as tests
import part3_mamba_from_scratch.utils as utils
from part3_mamba_from_scratch.solutions import MambaConfig, scan_equivalence_report


In [ ]:
def make_scan_inputs(seed: int = 0):
    t.manual_seed(seed)
    batch, seq, d_inner, d_state = 2, 7, 4, 3
    u = t.randn(batch, seq, d_inner)
    delta = t.rand(batch, seq, d_inner) + 0.1
    A_log = t.randn(d_inner, d_state) - 2.0
    B = t.randn(batch, seq, d_state)
    C = t.randn(batch, seq, d_state)
    D = t.randn(d_inner)
    z = t.randn(batch, seq, d_inner)
    return u, delta, A_log, B, C, D, z


## Selective scan

Implement the recurrence coefficients, the recurrent reference path, and the associative scan.

In [ ]:
def discretize_selective_scan(u: t.Tensor, delta: t.Tensor, A_log: t.Tensor, B: t.Tensor):
    raise NotImplementedError()


def selective_scan_recurrent(u, delta, A_log, B, C, D=None, z=None, initial_state=None, return_last_state=False):
    raise NotImplementedError()


def selective_scan_parallel(u, delta, A_log, B, C, D=None, z=None, initial_state=None, return_last_state=False):
    raise NotImplementedError()


tests.test_discretize_selective_scan_shapes_and_stability(discretize_selective_scan)
tests.test_selective_scan_single_step_manual(selective_scan_recurrent)
tests.test_recurrent_scan_matches_reference(selective_scan_recurrent)


## Chunked scan

Chunking should only change scheduling. It must not reset recurrent state at chunk boundaries.

In [ ]:
def selective_scan_chunked(u, delta, A_log, B, C, D=None, z=None, chunk_size=64, initial_state=None, return_last_state=False):
    raise NotImplementedError()


u, delta, A_log, B, C, D, z = make_scan_inputs()
full = selective_scan_recurrent(u, delta, A_log, B, C, D=D, z=z)
chunked = selective_scan_chunked(u, delta, A_log, B, C, D=D, z=z, chunk_size=3)
report = scan_equivalence_report(full, chunked, atol=1e-6)
utils.print_report("Full vs chunked scan", report.__dict__)
assert report.passed, report


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
